In [2]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/food-pipeline"
MODEL_SAVE_PATH = f"{DRIVE_PATH}/models/classifier"
DATA_PATH = f"{DRIVE_PATH}/data/food101"

import os
os.makedirs(MODEL_SAVE_PATH, exist_ok=True)
os.makedirs(DATA_PATH, exist_ok=True)
print("Drive mounted and paths ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive mounted and paths ready


In [10]:
!pip install -q timm safetensors huggingface_hub wandb datasets

In [4]:
import torch
print(f"GPU available: {torch.cuda.is_available()}")
print(f"GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

GPU available: True
GPU name: Tesla T4


In [5]:
from huggingface_hub import hf_hub_download
import json

hf_hub_download(
    repo_id="Lumia101/Food101-EfficientNet-B0",
    filename="config.json",
    local_dir=MODEL_SAVE_PATH
)
hf_hub_download(
    repo_id="Lumia101/Food101-EfficientNet-B0",
    filename="model.safetensors",
    local_dir=MODEL_SAVE_PATH
)
print("Base model downloaded")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Base model downloaded


In [11]:
from datasets import load_dataset
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
from PIL import Image

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

class HFFood101(Dataset):
    def __init__(self, hf_dataset, transform):
        self.data = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = item["image"].convert("RGB")
        label = item["label"]
        return self.transform(img), label

print("Loading Food-101 from Hugging Face...")
hf_data = load_dataset("ethz/food101")

train_dataset = HFFood101(hf_data["train"], train_transform)
val_dataset   = HFFood101(hf_data["validation"], val_transform)

train_loader = DataLoader(train_dataset, batch_size=64,
                          shuffle=True, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset, batch_size=64,
                          shuffle=False, num_workers=2, pin_memory=True)

class_names = hf_data["train"].features["label"].names

print(f"Train: {len(train_dataset)} images")
print(f"Val:   {len(val_dataset)} images")
print(f"Classes: {len(class_names)}")

Loading Food-101 from Hugging Face...


data/train-00000-of-00008.parquet:   0%|          | 0.00/490M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/472M [00:00<?, ?B/s]

data/train-00003-of-00008.parquet:   0%|          | 0.00/464M [00:00<?, ?B/s]

...

data/validation-00001-of-00003.parquet:   0%|          | 0.00/413M [00:00<?, ?B/s]

data/validation-00002-of-00003.parquet:   0%|          | 0.00/426M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/75750 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25250 [00:00<?, ? examples/s]

Train: 75750 images
Val:   25250 images
Classes: 101


In [12]:
import torch
import torch.nn as nn
import torchvision.models as models
from safetensors.torch import load_file

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.efficientnet_b0(weights=None)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(1280, 512),
    nn.SiLU(),
    nn.Dropout(0.2),
    nn.Linear(512, 101)
)

state_dict = load_file(f"{MODEL_SAVE_PATH}/model.safetensors")
model.load_state_dict(state_dict)
model = model.to(device)
print(f"Model loaded on {device}")

Model loaded on cuda


In [13]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
import time

optimizer = AdamW([
    {"params": model.features.parameters(), "lr": 5e-5},
    {"params": model.classifier.parameters(), "lr": 2e-4}
])
scheduler = CosineAnnealingLR(optimizer, T_max=10)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct += outputs.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
    return total_loss / len(loader), correct / total

def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            correct += outputs.argmax(1).eq(labels).sum().item()
            total += labels.size(0)
    return total_loss / len(loader), correct / total

EPOCHS = 10
best_acc = 0.7957  # start tracking above Lumia101 baseline

for epoch in range(EPOCHS):
    t0 = time.time()
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = val_epoch(model, val_loader, criterion, device)
    scheduler.step()

    elapsed = time.time() - t0
    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"train_loss={train_loss:.3f} acc={train_acc:.3f} | "
          f"val_loss={val_loss:.3f} acc={val_acc:.3f} | "
          f"{elapsed:.0f}s")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(),
                   f"{MODEL_SAVE_PATH}/efficientnet_b0_food101_best.pt")
        print(f"  Saved new best model (acc={best_acc:.4f})")

/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 01/10 | train_loss=1.727 acc=0.735 | val_loss=1.321 acc=0.856 | 691s
  Saved new best model (acc=0.8564)


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 02/10 | train_loss=1.660 acc=0.756 | val_loss=1.308 acc=0.859 | 692s
  Saved new best model (acc=0.8588)


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


...

Epoch 08/10 | train_loss=1.511 acc=0.801 | val_loss=1.261 acc=0.871 | 676s
  Saved new best model (acc=0.8708)


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 09/10 | train_loss=1.505 acc=0.804 | val_loss=1.257 acc=0.872 | 684s
  Saved new best model (acc=0.8722)


/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.12/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))


Epoch 10/10 | train_loss=1.502 acc=0.806 | val_loss=1.257 acc=0.874 | 685s
  Saved new best model (acc=0.8737)


In [14]:
import json

idx_to_class = {i: name for i, name in enumerate(class_names)}

with open(f"{MODEL_SAVE_PATH}/idx_to_class.json", "w") as f:
    json.dump(idx_to_class, f, indent=2)

print(f"Saved {len(idx_to_class)} class labels")
print("Sample:", list(idx_to_class.items())[:5])

Saved 101 class labels
Sample: [(0, 'apple_pie'), (1, 'baby_back_ribs'), (2, 'baklava'), (3, 'beef_carpaccio'), (4, 'beef_tartare')]


In [17]:
from google.colab import files

files.download(f"{MODEL_SAVE_PATH}/efficientnet_b0_food101_best.pt")
files.download(f"{MODEL_SAVE_PATH}/idx_to_class.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
import os
print(os.path.exists(f"{MODEL_SAVE_PATH}/efficientnet_b0_food101_best.pt"))
print(os.path.exists(f"{MODEL_SAVE_PATH}/idx_to_class.json"))
print(MODEL_SAVE_PATH)

True
True
/content/drive/MyDrive/food-pipeline/models/classifier
